# Fine-tuning nanoVLM for MiniGrid EmptyEnv Action Prediction

This notebook implements the required FT baseline for a vision-language agent in `MiniGrid-EmptyEnv`.

The goal is to fine-tune `nanoVLM` so that, given the current visual observation, it predicts the next discrete action needed to reach the goal.

**Action space used in this notebook**

| Action id | Action |
|---:|---|
| 0 | turn left |
| 1 | turn right |
| 2 | move forward |

The notebook contains four parts: environment setup, dataset loading, supervised fine-tuning, and evaluation through learning curves and validation metrics.


## 1. Expert Dataset

The dataset consists of expert trajectories collected in `MiniGrid-EmptyEnv`. Each sample contains an RGB observation and the expert's next action.

The expert policy was implemented as a shortest-path planner. This is a natural choice for `EmptyEnv` because the map contains no obstacles: the shortest path to the goal is well-defined, deterministic, and provides clean action labels. At each state, the expert aligns the agent with the direction of the next shortest-path step and then moves forward.

The final supervised fine-tuning dataset is published on Hugging Face as `arinamir/emptyEnv` and is loaded below.


## 2. Environment Setup

These cells clone the `nanoVLM` repository, install the required dependencies, authenticate with Hugging Face when needed, and import the training utilities.


In [ ]:
import os
import subprocess

repo_path = "/content/nanoVLM"

if not os.path.isdir(repo_path):
    subprocess.run(
        ["git", "clone", "https://github.com/huggingface/nanoVLM.git", repo_path],
        check=True,
    )

%cd /content/nanoVLM


In [ ]:
from pathlib import Path

processors_path = Path("data/processors.py")

old_get_tokenizer = '''def get_tokenizer(name, extra_special_tokens=None, chat_template=None):
    if name not in TOKENIZERS_CACHE:
        tokenizer_init_kwargs = {"use_fast": True}
        if extra_special_tokens is not None:
            tokenizer_init_kwargs["extra_special_tokens"] = extra_special_tokens
        if chat_template is not None:
            tokenizer_init_kwargs["chat_template"] = chat_template
        tokenizer = AutoTokenizer.from_pretrained(name, **tokenizer_init_kwargs,)
        tokenizer.pad_token = tokenizer.eos_token
        TOKENIZERS_CACHE[name] = tokenizer
    return TOKENIZERS_CACHE[name]
'''

new_get_tokenizer = '''def get_tokenizer(name, extra_special_tokens=None, chat_template=None):
    if name not in TOKENIZERS_CACHE:
        tokenizer = AutoTokenizer.from_pretrained(name, use_fast=True)

        if extra_special_tokens is not None:
            new_tokens = list(extra_special_tokens.values())
            tokenizer.add_tokens(new_tokens, special_tokens=True)

            for token_name, token_str in extra_special_tokens.items():
                setattr(tokenizer, token_name, token_str)
                token_id = tokenizer.convert_tokens_to_ids(token_str)
                setattr(tokenizer, token_name + "_id", token_id)

        if chat_template is not None:
            tokenizer.chat_template = chat_template

        tokenizer.pad_token = tokenizer.eos_token
        TOKENIZERS_CACHE[name] = tokenizer

    return TOKENIZERS_CACHE[name]
'''

source = processors_path.read_text()

if old_get_tokenizer not in source:
    raise RuntimeError(
        "Original get_tokenizer implementation was not found. "
        "The repository file may have changed."
    )

source = source.replace(old_get_tokenizer, new_get_tokenizer)

processors_path.write_text(source)

print("Patched data/processors.py:get_tokenizer")

In [ ]:
# The dependency resolver may print warnings in Colab; they are usually safe if the cell finishes successfully.
!pip install -q transformers==4.45.0
!pip install -q torch gcsfs tqdm huggingface_hub
!pip install -q datasets==3.5.0


In [ ]:
# Run this cell only if you need to access a private dataset or push a checkpoint.
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
# Change this to your own Hugging Face repository before pushing a checkpoint.
hf_model_name = "arinamir/nanoVLM"


In [ ]:
# nanoVLM components used for dataset loading, collation, preprocessing, and modeling.
from data.datasets import VQADataset
from data.collators import VQACollator
from data.data_utils import synchronized_dataloader_step
from data.advanced_datasets import ConstantLengthDataset
from data.processors import get_image_processor, get_tokenizer

import models.config as config
from models.vision_language_model import VisionLanguageModel

# Standard libraries and training dependencies.
import math
import time
import torch
from tqdm import tqdm
import torch.optim as optim
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from torch.utils.data import DataLoader
from datasets import load_dataset, concatenate_datasets, get_dataset_config_names

# Avoid tokenizer parallelism warnings in notebook environments.
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

# Uncomment these lines while developing nanoVLM modules locally.
# %reload_ext autoreload
# %autoreload 2


## 3. Optional Google Drive Checkpoint Backup

Mounting Google Drive is optional. It is useful in Colab when checkpoints should be preserved after the runtime is disconnected.


In [ ]:
# Optional: mount Google Drive to copy checkpoints after training.
from google.colab import drive

drive.mount('/content/drive')


## 4. Dataset Inspection

We load the expert trajectory dataset and verify that Hugging Face returns the expected split structure.


In [ ]:
# Inspect the published dataset used for supervised fine-tuning.
dataset = load_dataset("arinamir/emptyEnvText")
print(dataset)


In [ ]:
example = dataset["train"][0]
print("Available fields:", example.keys())
print("Text format:", example["texts"])


## 5. Data Loaders

The data loader converts the Hugging Face dataset into the `VQADataset` format expected by `nanoVLM`. The observation image is processed by the vision encoder pipeline, while the target action is represented as the assistant answer in the chat template.


In [ ]:
def get_dataloaders(train_cfg, vlm_cfg):
    # Create datasets
    image_processor = get_image_processor(vlm_cfg.max_img_size, vlm_cfg.vit_img_size, vlm_cfg.resize_to_max_side_len)
    tokenizer = get_tokenizer(vlm_cfg.lm_tokenizer, vlm_cfg.vlm_extra_tokens, vlm_cfg.lm_chat_template)

    # Load and combine all training datasets
    dataset_names_to_load = train_cfg.train_dataset_name
    if "all" in dataset_names_to_load:
        dataset_names_to_load = get_dataset_config_names(train_cfg.train_dataset_path)

    combined_train_data = []

    for dataset_name in dataset_names_to_load:
        print(f"Loading dataset: {dataset_name}")
        try:
            train_ds = load_dataset(train_cfg.train_dataset_path, dataset_name)['train']
            train_ds[0]  # Verify that the dataset config is readable.
            combined_train_data.append(train_ds)
        except Exception as e:
            print(f"Warning: Failed to load dataset config '{dataset_name}' from '{train_cfg.train_dataset_path}'. Error: {e}")
            continue
    train_ds = concatenate_datasets(combined_train_data)

    train_ds = train_ds.shuffle(seed=0)  # Mix samples before creating the train/validation split.

    # Apply cutoff if specified
    if train_cfg.data_cutoff_idx is None:
        total_samples = len(train_ds)  # Use the entire dataset
    else:
        total_samples = min(len(train_ds), train_cfg.data_cutoff_idx)

    val_size = int(total_samples * train_cfg.val_ratio)
    train_size = total_samples - val_size

    val_ds = train_ds.select(range(train_size, total_samples-1))
    train_ds = train_ds.select(range(train_size))

    train_dataset = VQADataset(train_ds, tokenizer, image_processor, vlm_cfg.mp_image_token_length)
    val_dataset = VQADataset(val_ds, tokenizer, image_processor, vlm_cfg.mp_image_token_length)

    train_dataset = ConstantLengthDataset(train_dataset, infinite=False, max_sample_length=train_cfg.max_sample_length, seq_length=vlm_cfg.lm_max_length, num_of_sequences=train_cfg.batch_size*4, queue_size=8,
                                        max_images_per_example=train_cfg.max_images_per_example, max_images_per_knapsack=train_cfg.max_images_per_knapsack)

    # Create collators
    vqa_collator = VQACollator(tokenizer, vlm_cfg.lm_max_length)

    # Create dataloaders

    train_loader = DataLoader(
        train_dataset,
        batch_size=train_cfg.batch_size,
        collate_fn=vqa_collator,
        num_workers=1,
        pin_memory=True,
        persistent_workers=True,
        drop_last=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=train_cfg.batch_size,
        shuffle=False,
        collate_fn=vqa_collator,
        num_workers=1,
        pin_memory=True,
        persistent_workers=True,
        drop_last=True,
    )

    # Warmup dataloaders to kickstart worker processes
    print("Warming up dataloaders...")
    next(iter(train_loader))
    next(iter(val_loader))
    print("Warmup complete.")

    return train_loader, val_loader


## 6. Training Loop

The training loop performs supervised fine-tuning of nanoVLM. The modality projection layer, vision backbone, and language backbone can use separate learning rates. Validation loss is computed periodically, and checkpoints are saved every 50 training steps so that learning progress can be evaluated later.


In [ ]:
def get_lr(it, max_lr, max_steps):
    min_lr = max_lr * 0.1
    warmup_steps = max_steps * 0.03
    # Linear warmup for the first 3% of training steps.
    if it < warmup_steps:
        return max_lr * (it+1) / warmup_steps
    # Keep the learning rate at its minimum after the configured training horizon.
    if it > max_steps:
        return min_lr
    # Cosine decay from the maximum to the minimum learning rate.
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # Goes from 1 to 0.
    return min_lr + coeff * (max_lr - min_lr)

def train(train_cfg, vlm_cfg):
    train_loader, val_loader = get_dataloaders(train_cfg, vlm_cfg)

    # Initialize model
    if train_cfg.resume_from_vlm_checkpoint:
        print(f"Resuming from VLM checkpoint: {vlm_cfg.vlm_checkpoint_path}")
        model = VisionLanguageModel.from_pretrained(vlm_cfg.vlm_checkpoint_path)
    else:
        model = VisionLanguageModel(vlm_cfg, load_backbone=vlm_cfg.vlm_load_backbone_weights)

    print(f"nanoVLM initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
    print(f"Training summary: {len(train_loader.dataset)} samples, {len(train_loader)} batches/epoch, batch size {train_cfg.batch_size}")

    # Define optimizer groups
    # The modality projection layer is newly initialized, while the vision and language backbones are pretrained.
    # We therefore use separate learning rates and optionally freeze parts of the model.
    param_groups = []
    if train_cfg.lr_mp > 0:
        param_groups.append({'params': list(model.MP.parameters()), 'lr': train_cfg.lr_mp})
    else:
        for p in list(model.MP.parameters()):
            p.requires_grad = False
    if train_cfg.lr_vision_backbone > 0:
        param_groups.append({'params': list(model.vision_encoder.parameters()), 'lr': train_cfg.lr_vision_backbone})
    else:
        for p in list(model.vision_encoder.parameters()):
            p.requires_grad = False
    if train_cfg.lr_language_backbone > 0:
        param_groups.append({'params': list(model.decoder.parameters()), 'lr': train_cfg.lr_language_backbone})
    else:
        for p in list(model.decoder.parameters()):
            p.requires_grad = False

    optimizer = optim.AdamW(param_groups)
    all_params = [p for group in optimizer.param_groups for p in group['params']]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    if train_cfg.compile:
        model = torch.compile(model)

    epoch_times = []
    batch_losses = []
    val_losses = []
    val_plot_steps = []
    global_step = 0
    epoch = 0

    while global_step < train_cfg.max_training_steps:
        epoch_start_time = time.time()
        epoch += 1
        model.train()
        total_train_loss = 0
        total_tokens_processed = 0
        optimizer.zero_grad()

        print("Starting training loop")
        for i, batch in enumerate(synchronized_dataloader_step(train_loader, False)):
            batch_start_time = time.time()
            is_update_step = (i + 1) % train_cfg.gradient_accumulation_steps == 0 or i + 1 == len(train_loader)
            images = batch["images"]
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            with torch.autocast(device_type='cuda', dtype=torch.float16):  # Mixed precision training.
                _, loss = model(input_ids, images, attention_mask=attention_mask, targets=labels)

            if train_cfg.gradient_accumulation_steps > 1:
                loss = loss / train_cfg.gradient_accumulation_steps

            loss.backward()

            if is_update_step:
                if train_cfg.max_grad_norm is not None:
                    _ = torch.nn.utils.clip_grad_norm_(all_params, max_norm=train_cfg.max_grad_norm)

                param_group_idx = 0
                if train_cfg.lr_mp > 0:
                    adj_lr_mp = get_lr(global_step, train_cfg.lr_mp, train_cfg.max_training_steps)
                    optimizer.param_groups[param_group_idx]['lr'] = adj_lr_mp
                    param_group_idx += 1

                if train_cfg.lr_vision_backbone > 0:
                    adj_lr_vision_backbone = get_lr(global_step, train_cfg.lr_vision_backbone, train_cfg.max_training_steps)
                    optimizer.param_groups[param_group_idx]['lr'] = adj_lr_vision_backbone
                    param_group_idx += 1

                if train_cfg.lr_language_backbone > 0:
                    adj_lr_language_backbone = get_lr(global_step, train_cfg.lr_language_backbone, train_cfg.max_training_steps)
                    optimizer.param_groups[param_group_idx]['lr'] = adj_lr_language_backbone

                optimizer.step()
                optimizer.zero_grad()

            batch_loss = loss.item()
            if train_cfg.gradient_accumulation_steps > 1:
                batch_loss = batch_loss * train_cfg.gradient_accumulation_steps
            total_train_loss += batch_loss
            batch_losses.append(batch_loss)

            num_tokens = torch.sum(attention_mask).item()  # Number of active tokens in the batch.
            total_tokens_processed += num_tokens

            batch_end_time = time.time()
            batch_duration = batch_end_time - batch_start_time
            tokens_per_second = num_tokens / batch_duration

            if global_step % 50 == 0:
                model.eval()
                torch.cuda.empty_cache()  # Release unused GPU memory before validation.
                with torch.no_grad():
                    total_val_loss = 0
                    for batch in synchronized_dataloader_step(val_loader, False):
                        images = batch["images"]
                        input_ids = batch["input_ids"].to(device)
                        labels = batch["labels"].to(device)
                        attention_mask = batch["attention_mask"].to(device)

                        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                            _, loss = model(input_ids, images, attention_mask=attention_mask, targets=labels)

                        total_val_loss += loss.item()
                    avg_val_loss = total_val_loss / len(val_loader)
                    val_losses.append(avg_val_loss)
                    val_plot_steps.append(global_step)
                print(f"\nStep: {global_step}, Loss: {batch_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Tokens/s: {tokens_per_second:.2f}")
                model.train()

            if global_step % 50 == 0:
                # Save a checkpoint for later evaluation.
                checkpoint_path = f"checkpoints/step_{global_step}"
                model.save_pretrained(checkpoint_path)

            global_step += 1

        avg_train_loss = total_train_loss / len(train_loader)

        epoch_end_time = time.time()
        epoch_duration = epoch_end_time - epoch_start_time
        epoch_times.append(epoch_duration)

        epoch_tokens_per_second = total_tokens_processed / epoch_duration

        print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {epoch_duration:.2f}s | T/s: {epoch_tokens_per_second:.2f}")

    model.save_pretrained(save_directory=vlm_cfg.vlm_checkpoint_path)
    model.push_to_hub(hf_model_name)

    total_training_time = sum(epoch_times)
    print(f"Total training time: {total_training_time:.2f}s")

    plt.plot(batch_losses, label='Train Loss')
    plt.plot(val_plot_steps, val_losses, label='Val Loss')
    plt.xlabel('Batch')
    plt.ylabel('Loss')
    plt.title('Loss Curve')
    plt.grid(True)
    plt.legend()
    plt.show()


## 7. Model and Training Configuration

The configuration is defined directly in the notebook so that it is easy to adapt for Colab. The dataset path points to the expert EmptyEnv dataset, and the maximum number of training steps controls the FT baseline budget.


In [ ]:
@dataclass
class VLMConfig:
    vit_hidden_dim: int = 768
    vit_inter_dim: int = 4 * vit_hidden_dim
    vit_patch_size: int = 16
    vit_img_size: int = 512
    vit_n_heads: int = 12
    vit_dropout: float = 0.0
    vit_n_blocks: int = 12
    vit_ln_eps: float = 1e-6
    vit_cls_flag: bool = False
    vit_model_type: str = 'google/siglip2-base-patch16-512'

    lm_hidden_dim: int = 960
    lm_inter_dim: int = 2560
    lm_rms_eps: float = 1e-5
    lm_re_base: int = 100000
    lm_max_position_embeddings: int = 8192
    lm_base_vocab_size: int = 49152
    extra_token_amount: int = 66  # Extra VLM tokens for image markers and image grid positions.
    lm_vocab_size: int = lm_base_vocab_size + extra_token_amount
    lm_n_heads: int = 15
    lm_n_kv_heads: int = 5
    lm_dropout: float = 0.0
    lm_n_blocks: int = 32
    lm_attn_scaling: float = 1.0
    lm_max_length: int = 1024
    lm_use_tokens: bool = False  # The language model receives embeddings inside the VLM.
    lm_tie_weights: bool = True  # Tie the LM head weights to token embeddings.
    lm_model_type: str = 'HuggingFaceTB/SmolLM2-135M'
    lm_tokenizer: str = 'HuggingFaceTB/SmolLM2-360M-Instruct'
    lm_chat_template: str = "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

    mp_pixel_shuffle_factor: int = 4
    mp_image_token_length: int = 64

    max_img_size: int = 512
    resize_to_max_side_len: bool = False

    vlm_extra_tokens: dict[str, str] = field(default_factory=lambda: {"image_token": "<|image|>", "global_image_token": "<|global_image|>",
      "r1c1": "<row_1_col_1>", "r1c2": "<row_1_col_2>", "r1c3": "<row_1_col_3>", "r1c4": "<row_1_col_4>", "r1c5": "<row_1_col_5>", "r1c6": "<row_1_col_6>", "r1c7": "<row_1_col_7>", "r1c8": "<row_1_col_8>",
      "r2c1": "<row_2_col_1>", "r2c2": "<row_2_col_2>", "r2c3": "<row_2_col_3>", "r2c4": "<row_2_col_4>", "r2c5": "<row_2_col_5>", "r2c6": "<row_2_col_6>", "r2c7": "<row_2_col_7>", "r2c8": "<row_2_col_8>",
      "r3c1": "<row_3_col_1>", "r3c2": "<row_3_col_2>", "r3c3": "<row_3_col_3>", "r3c4": "<row_3_col_4>", "r3c5": "<row_3_col_5>", "r3c6": "<row_3_col_6>", "r3c7": "<row_3_col_7>", "r3c8": "<row_3_col_8>",
      "r4c1": "<row_4_col_1>", "r4c2": "<row_4_col_2>", "r4c3": "<row_4_col_3>", "r4c4": "<row_4_col_4>", "r4c5": "<row_4_col_5>", "r4c6": "<row_4_col_6>", "r4c7": "<row_4_col_7>", "r4c8": "<row_4_col_8>",
      "r5c1": "<row_5_col_1>", "r5c2": "<row_5_col_2>", "r5c3": "<row_5_col_3>", "r5c4": "<row_5_col_4>", "r5c5": "<row_5_col_5>", "r5c6": "<row_5_col_6>", "r5c7": "<row_5_col_7>", "r5c8": "<row_5_col_8>",
      "r6c1": "<row_6_col_1>", "r6c2": "<row_6_col_2>", "r6c3": "<row_6_col_3>", "r6c4": "<row_6_col_4>", "r6c5": "<row_6_col_5>", "r6c6": "<row_6_col_6>", "r6c7": "<row_6_col_7>", "r6c8": "<row_6_col_8>",
      "r7c1": "<row_7_col_1>", "r7c2": "<row_7_col_2>", "r7c3": "<row_7_col_3>", "r7c4": "<row_7_col_4>", "r7c5": "<row_7_col_5>", "r7c6": "<row_7_col_6>", "r7c7": "<row_7_col_7>", "r7c8": "<row_7_col_8>",
      "r8c1": "<row_8_col_1>", "r8c2": "<row_8_col_2>", "r8c3": "<row_8_col_3>", "r8c4": "<row_8_col_4>", "r8c5": "<row_8_col_5>", "r8c6": "<row_8_col_6>", "r8c7": "<row_8_col_7>", "r8c8": "<row_8_col_8>"})
    vlm_load_backbone_weights: bool = True
    vlm_checkpoint_path: str = 'checkpoints'
    hf_repo_name: str = 'nanoVLM'


@dataclass
class TrainConfig:
    lr_mp: float = 0.005
    lr_vision_backbone: float = 0.0005
    lr_language_backbone: float = 0.0005
    data_cutoff_idx: int = 5000  # Use a subset for the baseline run.
    val_ratio: float = 0.2
    batch_size: int = 1
    gradient_accumulation_steps: int = 4
    max_grad_norm: float = 1.0
    max_training_steps: int = 500
    max_images_per_example: int = 2
    max_images_per_knapsack: int = 8
    max_sample_length: int = 1024
    compile: bool = False
    resume_from_vlm_checkpoint: bool = False  # Set to True to resume from a full VLM checkpoint.
    train_dataset_path: str = 'arinamir/emptyEnvText'
    train_dataset_name: tuple[str, ...] = ("default",)


## 8. Run Fine-Tuning

This cell starts supervised fine-tuning. During training, the notebook logs train loss, validation loss, and throughput. The final checkpoint can optionally be pushed to the Hugging Face Hub.


In [ ]:
vlm_cfg = VLMConfig()
train_cfg = TrainConfig()
train(train_cfg, vlm_cfg)


## 8. Save checkpoints

After training, copy checkpoints to persistent storage if needed.


In [ ]:
# Optional: back up checkpoints to Google Drive.
!cp -r /content/nanoVLM/checkpoints /content/drive/MyDrive/


## 9. Rollout evaluation: action-only output

To evaluate the trained model as a policy, selected checkpoints are rolled out in `MiniGrid-Empty-Random-6x6-v0`. For each checkpoint, the notebook reports success rate and average return over a fixed set of seeds, then plots both learning curves.


In [ ]:
!pip install -q minigrid


In [ ]:
import os
import gc
import csv
import torch
import gymnasium as gym
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from minigrid.wrappers import RGBImgObsWrapper

from models.vision_language_model import VisionLanguageModel
from data.processors import (
    get_tokenizer,
    get_image_processor,
    get_image_string
)

# Evaluation configuration

CHECKPOINT_DIR = "/content/nanoVLM/checkpoints"

SELECTED_STEPS = [
    0,
    100,
    150,
    200,
    250,
    300
]

NUM_EPISODES = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EVAL_SEEDS = list(range(NUM_EPISODES))

# Action prediction helper

def get_action_from_model(
        model,
        obs_image,
        tokenizer,
        image_processor):

    img = Image.fromarray(obs_image).convert("RGB")

    processed_image, ratio = image_processor(img)

    if processed_image.dim() == 5:
        img_t = processed_image.squeeze(1).to(DEVICE)

    elif processed_image.dim() == 4:
        img_t = processed_image.to(DEVICE)

    elif processed_image.dim() == 3:
        img_t = processed_image.unsqueeze(0).to(DEVICE)

    else:
        raise ValueError(
            f"Unexpected image shape: {processed_image.shape}"
        )

    image_string = get_image_string(
        tokenizer,
        [ratio],
        model.cfg.mp_image_token_length
    )

    # Use the same prompt format as in the training dataset.
    prompt = "What should the agent do next to reach the goal?"

    user_text = image_string + prompt

    messages = [
        {
            "role": "user",
            "content": user_text
        }
    ]

    tokens = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True
    )

    tokens = (
        torch.tensor(tokens)
        .unsqueeze(0)
        .to(DEVICE)
    )

    with torch.inference_mode():

        generated = model.generate(
            tokens,
            img_t,
            max_new_tokens=3
        )

    # Decode only the newly generated answer tokens.

    input_len = tokens.shape[1]

    answer_tokens = generated[0][input_len:]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip().lower()

    # print(answer)

    if answer.startswith("left"):
        return 0

    if answer.startswith("right"):
        return 1

    if answer.startswith("forward"):
        return 2

    # fallback
    return 2


# Checkpoint evaluation

def evaluate_checkpoint(
        checkpoint_path,
        env,
        tokenizer,
        image_processor):

    print(f"\nLoading {checkpoint_path}")

    model = (
        VisionLanguageModel
        .from_pretrained(checkpoint_path)
        .to(DEVICE)
    )

    model.eval()

    successes = 0
    returns = []

    for seed in tqdm(
            EVAL_SEEDS,
            desc="Evaluation"):

        obs, _ = env.reset(seed=seed)

        terminated = False
        truncated = False

        episode_return = 0

        while not (terminated or truncated):

            action = get_action_from_model(
                model,
                obs["image"],
                tokenizer,
                image_processor
            )

            obs, reward, terminated, truncated, _ = env.step(action)

            episode_return += reward

        if reward > 0:
            successes += 1

        returns.append(episode_return)

    success_rate = successes / len(EVAL_SEEDS)

    mean_return = sum(returns) / len(returns)

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return success_rate, mean_return


# Environment

env = gym.make(
    "MiniGrid-Empty-Random-6x6-v0",
    render_mode="rgb_array"
)

env = RGBImgObsWrapper(env)

# Load tokenizer and image processor from the first checkpoint

first_checkpoint = os.path.join(
    CHECKPOINT_DIR,
    "step_0"
)

tmp_model = VisionLanguageModel.from_pretrained(
    first_checkpoint
)

tokenizer = get_tokenizer(
    tmp_model.cfg.lm_tokenizer,
    tmp_model.cfg.vlm_extra_tokens,
    tmp_model.cfg.lm_chat_template
)

image_processor = get_image_processor(
    tmp_model.cfg.max_img_size,
    tmp_model.cfg.vit_img_size,
    tmp_model.cfg.resize_to_max_side_len
)

del tmp_model
gc.collect()

# Evaluate selected checkpoints

results = []

for step in SELECTED_STEPS:

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"step_{step}"
    )

    sr, mean_return = evaluate_checkpoint(
        checkpoint_path,
        env,
        tokenizer,
        image_processor
    )

    results.append(
        {
            "step": step,
            "success_rate": sr,
            "mean_return": mean_return
        }
    )

    print(
        f"Step {step}: "
        f"SR={sr:.3f}, "
        f"Return={mean_return:.3f}"
    )

# Save evaluation results

with open(
    "sft_evaluation.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "step",
            "success_rate",
            "mean_return"
        ]
    )

    writer.writeheader()

    for row in results:
        writer.writerow(row)

# Plot success rate

steps = [r["step"] for r in results]
srs = [r["success_rate"] for r in results]

plt.figure(figsize=(8, 5))

plt.plot(
    steps,
    srs,
    marker="o",
    linewidth=2
)

plt.xlabel("Training Step")
plt.ylabel("Success Rate")
plt.title("SFT Learning Curve")
plt.grid(True)

plt.show()

# Plot average return

returns = [r["mean_return"] for r in results]

plt.figure(figsize=(8, 5))

plt.plot(
    steps,
    returns,
    marker="o",
    linewidth=2
)

plt.xlabel("Training Step")
plt.ylabel("Average Return")
plt.title("Average Return per Checkpoint")
plt.grid(True)

plt.show()


## 10. Offline action-prediction evaluation

In addition to rollout evaluation, this section measures how accurately the model predicts the expert action on a held-out subset of dataset samples. It reports overall accuracy, per-action accuracy, balanced accuracy, and a confusion matrix.


In [ ]:
import re
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from PIL import Image
from sklearn.metrics import confusion_matrix

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor, get_image_string


CHECKPOINT_PATH = "/content/nanoVLM/checkpoints/step_1200"
DATASET_NAME = "arinamir/emptyEnv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_VAL = 200

PROMPT = """Look at the image. Action IDs:0 = left,1 = right,2 = forward. Choose the best action to reach the goal. Answer with only one number."""


LABELS = ["0", "1", "2"]
ID_TO_ACTION = {
    "0": "left",
    "1": "right",
    "2": "forward",
}


def parse_action(text):
    text = text.strip()

    # Use the first valid action id produced by the model.
    match = re.search(r"[012]", text)

    if match is None:
        return "invalid"

    return match.group(0)


def extract_gt_action(texts):
    if isinstance(texts, str):
        texts = json.loads(texts)

    action = texts[0]["assistant"].strip()

    if action in LABELS:
        return action

    return "invalid"


def predict_action(model, tokenizer, image_processor, image):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert("RGB")
    else:
        image = image.convert("RGB")

    processed_image, splitted_image_ratio = image_processor(image)

    if (
        not hasattr(tokenizer, "global_image_token")
        and splitted_image_ratio[0] * splitted_image_ratio[1] == len(processed_image) - 1
    ):
        processed_image = processed_image[1:]

    image_string = get_image_string(
        tokenizer,
        [splitted_image_ratio],
        model.cfg.mp_image_token_length
    )

    messages = [
        {
            "role": "user",
            "content": image_string + PROMPT
        }
    ]

    encoded_prompt = tokenizer.apply_chat_template(
        [messages],
        tokenize=True,
        add_generation_prompt=True
    )

    tokens = torch.tensor(encoded_prompt).to(DEVICE)
    img_t = processed_image.to(DEVICE)

    with torch.inference_mode():
        gen = model.generate(
            tokens,
            img_t,
            max_new_tokens=3
        )

    output_text = tokenizer.batch_decode(
        gen,
        skip_special_tokens=True
    )[0]

    return parse_action(output_text)


model = VisionLanguageModel.from_pretrained(CHECKPOINT_PATH).to(DEVICE)
model.eval()

tokenizer = get_tokenizer(
    model.cfg.lm_tokenizer,
    model.cfg.vlm_extra_tokens,
    model.cfg.lm_chat_template
)

resize_to_max_side_len = False
if hasattr(model.cfg, "resize_to_max_side_len"):
    resize_to_max_side_len = model.cfg.resize_to_max_side_len

image_processor = get_image_processor(
    model.cfg.max_img_size,
    model.cfg.vit_img_size,
    resize_to_max_side_len
)


ds_dict = load_dataset(DATASET_NAME)
ds = ds_dict["train"]

val_ds = ds.shuffle(seed=42).select(range(N_VAL))

y_true = []
y_pred = []

for sample in val_ds:
    gt_action = extract_gt_action(sample["texts"])

    pred_action = predict_action(
        model=model,
        tokenizer=tokenizer,
        image_processor=image_processor,
        image=sample["images"]
    )

    y_true.append(gt_action)
    y_pred.append(pred_action)


y_true = np.array(y_true)
y_pred = np.array(y_pred)

overall_acc = np.mean(y_true == y_pred)

print(f"Overall accuracy: {overall_acc:.4f}")

balanced_acc_values = []

for label in LABELS:
    mask = y_true == label

    if mask.sum() == 0:
        print(f"Accuracy {label} ({ID_TO_ACTION[label]}): no samples")
    else:
        acc = np.mean(y_pred[mask] == label)
        balanced_acc_values.append(acc)

        print(
            f"Accuracy {label} ({ID_TO_ACTION[label]}): "
            f"{acc:.4f} ({mask.sum()} samples)"
        )

balanced_acc = np.mean(balanced_acc_values)
print(f"Balanced accuracy: {balanced_acc:.4f}")


all_labels = LABELS + ["invalid"]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=all_labels
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"GT {x} ({ID_TO_ACTION.get(x, x)})"
        for x in all_labels
    ],
    columns=[
        f"Pred {x} ({ID_TO_ACTION.get(x, x)})"
        for x in all_labels
    ]
)

print(cm_df)


plt.figure(figsize=(7, 6))
plt.imshow(cm, aspect="auto")
plt.xticks(
    range(len(all_labels)),
    [f"{x}\n{ID_TO_ACTION.get(x, x)}" for x in all_labels],
    rotation=0
)
plt.yticks(
    range(len(all_labels)),
    [f"{x}\n{ID_TO_ACTION.get(x, x)}" for x in all_labels]
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.title("Confusion Matrix: Action IDs")
plt.colorbar()
plt.tight_layout()
plt.show()


## 11. Rollout evaluation: text + action output


In [ ]:
import os
import re
import gc
import csv
import torch
import gymnasium as gym
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from minigrid.wrappers import RGBImgObsWrapper

from models.vision_language_model import VisionLanguageModel
from data.processors import (
    get_tokenizer,
    get_image_processor,
    get_image_string
)


# =====================================================
# CONFIG
# =====================================================

CHECKPOINT_DIR = "/content/nanoVLM/checkpoints"

SELECTED_STEPS = [
    0,
    100,
    250,
    400,
    800
]

NUM_EPISODES = 10
MAX_EPISODE_STEPS = 30

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EVAL_SEEDS = list(range(NUM_EPISODES))

TILE_SIZE = 42

PROMPT = "Look at the image.\n\nDescribe the current situation in 1-2 short sentences.\nThen choose the best action to reach the goal.\n\nAction IDs:\n0 = left\n1 = right\n2 = forward\n\nUse exactly this format:\nState: ...\nPlan: ...\nAction: 0/1/2"

# =====================================================
# PARSING
# =====================================================

def parse_action_from_text(text):
    if text is None:
        return 2, "fallback_forward"

    text = text.strip()

    # Expected primary format: Action: 0 / Action: 1 / Action: 2
    match = re.search(r"Action\s*:\s*([012])", text, flags=re.IGNORECASE)

    if match is not None:
        action = int(match.group(1))
        return action, f"action_{action}"

    # Fallback: use the last action id found in the response.
    matches = re.findall(r"[012]", text)

    if len(matches) > 0:
        action = int(matches[-1])
        return action, f"fallback_digit_{action}"

    return 2, "fallback_forward"


# =====================================================
# TOKENIZER / IMAGE PROCESSOR
# =====================================================

def prepare_processors(model):
    tokenizer = get_tokenizer(
        model.cfg.lm_tokenizer,
        model.cfg.vlm_extra_tokens,
        model.cfg.lm_chat_template
    )

    resize_to_max_side_len = False
    if hasattr(model.cfg, "resize_to_max_side_len"):
        resize_to_max_side_len = model.cfg.resize_to_max_side_len

    image_processor = get_image_processor(
        model.cfg.max_img_size,
        model.cfg.vit_img_size,
        resize_to_max_side_len
    )

    return tokenizer, image_processor


# =====================================================
# MODEL ACTION
# =====================================================

def get_action_from_model(
        model,
        obs_image,
        tokenizer,
        image_processor):

    img = Image.fromarray(obs_image).convert("RGB")

    processed_image, ratio = image_processor(img)

    if processed_image.dim() == 5:
        img_t = processed_image.squeeze(1).to(DEVICE)

    elif processed_image.dim() == 4:
        img_t = processed_image.to(DEVICE)

    elif processed_image.dim() == 3:
        img_t = processed_image.unsqueeze(0).to(DEVICE)

    else:
        raise ValueError(
            f"Unexpected image shape: {processed_image.shape}"
        )

    image_string = get_image_string(
        tokenizer,
        [ratio],
        model.cfg.mp_image_token_length
    )

    user_text = image_string + PROMPT

    messages = [
        {
            "role": "user",
            "content": user_text
        }
    ]

    encoded_prompt = tokenizer.apply_chat_template(
        [messages],
        tokenize=True,
        add_generation_prompt=True
    )

    tokens = torch.tensor(
        encoded_prompt,
        device=DEVICE
    )

    if tokens.dim() == 1:
        tokens = tokens.unsqueeze(0)

    with torch.inference_mode():
        generated = model.generate(
            tokens,
            img_t,
            max_new_tokens=80
        )

    # First try to decode only newly generated tokens.
    
    input_len = tokens.shape[1]
    answer_tokens = generated[0][input_len:]

    answer = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip()

    # If the answer is empty, decode the full generated output.
    if len(answer) == 0:
        answer = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )[0].strip()

    action, parsed = parse_action_from_text(answer)

    return action, parsed, answer


# =====================================================
# EVALUATION
# =====================================================

def evaluate_checkpoint(
        checkpoint_path,
        env,
        tokenizer,
        image_processor,
):


    print(f"\nLoading {checkpoint_path}")

    model = (
        VisionLanguageModel
        .from_pretrained(checkpoint_path)
        .to(DEVICE)
    )

    model.eval()

    successes = 0
    returns = []
    lengths = []

    invalid_outputs = 0
    fallback_outputs = 0

    for ep_idx, seed in enumerate(
            tqdm(EVAL_SEEDS, desc="Evaluation")):

        obs, _ = env.reset(seed=seed)

        terminated = False
        truncated = False

        episode_return = 0.0
        episode_steps = 0
        success = False

        while (
            not terminated
            and not truncated
            and episode_steps < MAX_EPISODE_STEPS
        ):

            action, parsed, answer = get_action_from_model(
                model,
                obs["image"],
                tokenizer,
                image_processor,
            )

            if parsed == "fallback_forward":
                invalid_outputs += 1

            if parsed.startswith("fallback"):
                fallback_outputs += 1

            obs, reward, terminated, truncated, _ = env.step(action)

            episode_return += reward
            episode_steps += 1

            if terminated and reward > 0:
                success = True

        if success:
            successes += 1

        returns.append(episode_return)
        lengths.append(episode_steps)

    success_rate = successes / len(EVAL_SEEDS)
    mean_return = sum(returns) / len(returns)
    mean_length = sum(lengths) / len(lengths)

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "success_rate": success_rate,
        "mean_return": mean_return,
        "mean_length": mean_length,
        "invalid_outputs": invalid_outputs,
        "fallback_outputs": fallback_outputs,
    }


# =====================================================
# ENV
# =====================================================

env = gym.make(
    "MiniGrid-Empty-Random-6x6-v0",
    render_mode="rgb_array"
)

env = RGBImgObsWrapper(
    env,
    tile_size=TILE_SIZE
)


# =====================================================
# LOAD TOKENIZER / IMAGE PROCESSOR
# =====================================================

first_checkpoint = os.path.join(
    CHECKPOINT_DIR,
    f"step_{SELECTED_STEPS[0]}"
)

tmp_model = VisionLanguageModel.from_pretrained(
    first_checkpoint
)

tokenizer, image_processor = prepare_processors(tmp_model)

del tmp_model
gc.collect()


# =====================================================
# MAIN LOOP
# =====================================================

results = []

for step in SELECTED_STEPS:

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"step_{step}"
    )

    if not os.path.exists(checkpoint_path):
        print(f"Skipping missing checkpoint: {checkpoint_path}")
        continue

    metrics = evaluate_checkpoint(
        checkpoint_path,
        env,
        tokenizer,
        image_processor,
    )

    row = {
        "step": step,
        **metrics
    }

    results.append(row)

    print(
        f"Step {step}: "
        f"SR={metrics['success_rate']:.3f}, "
        f"Return={metrics['mean_return']:.3f}, "
        f"Length={metrics['mean_length']:.2f}, "
        f"Invalid={metrics['invalid_outputs']}, "
        f"Fallback={metrics['fallback_outputs']}"
    )


# =====================================================
# SAVE CSV
# =====================================================

with open(
    "sft_text_action_evaluation.csv",
    "w",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "step",
            "success_rate",
            "mean_return",
            "mean_length",
            "invalid_outputs",
            "fallback_outputs",
        ]
    )

    writer.writeheader()
    writer.writerows(results)


# =====================================================
# PLOTS
# =====================================================

steps = [r["step"] for r in results]
srs = [r["success_rate"] for r in results]
returns = [r["mean_return"] for r in results]

plt.figure(figsize=(8, 5))
plt.plot(steps, srs, marker="o", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Success Rate")
plt.title("SFT Text+Action Success Rate")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(steps, returns, marker="o", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Average Return")
plt.title("SFT Text+Action Average Return")
plt.grid(True)
plt.tight_layout()
plt.show()
